In [1]:
cache_dir = "/mnt/labstore/bespoke_olap/cache/shell"

In [ ]:
# file_name = "65877adc24989db76bb1f4e789b4567a018dc84d6e23bb17e4113e9e01cccce7.pkl"

# # load this file and print a size summary of the contents
# import os
# import pickle

# file_path = os.path.join(cache_dir, file_name)
# with open(file_path, "rb") as f:
#     data = pickle.load(f)

# print(f"Loaded data from {file_path}")
# print(f"Data type: {type(data)}")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/labstore/bespoke_olap/cache/shell/65877adc24989db76bb1f4e789b4567a018dc84d6e23bb17e4113e9e01cccce7.pkl'

In [ ]:
# print(data.outputs[0].__dict__.keys())

dict_keys(['stdout', 'stderr', 'outcome', 'command', 'provider_data'])


In [3]:
import os
import pickle
import sys
from datetime import datetime

import pandas as pd


def deep_getsizeof(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(
            deep_getsizeof(k, seen) + deep_getsizeof(v, seen) for k, v in obj.items()
        )
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_getsizeof(i, seen) for i in obj)
    elif hasattr(obj, "__dict__"):
        size += deep_getsizeof(vars(obj), seen)

    return size


def fmt_bytes(n):
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    n = float(n)
    while n >= 1024 and i < len(units) - 1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"


def _normalize_command_value(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []
    if isinstance(value, (list, tuple, set)):
        out = []
        for item in value:
            out.extend(_normalize_command_value(item))
        return out
    return [str(value)]


def extract_commands(obj):
    found = []

    # Common top-level fields
    for attr in ("command", "cmd", "input"):
        found.extend(_normalize_command_value(getattr(obj, attr, None)))

    # Nested lists often contain one command per output/input object
    for seq_attr in ("outputs", "inputs"):
        seq = getattr(obj, seq_attr, None)
        if isinstance(seq, (list, tuple)):
            for item in seq:
                for attr in ("command", "cmd", "input"):
                    found.extend(_normalize_command_value(getattr(item, attr, None)))

    # Deduplicate while preserving order
    deduped = []
    seen = set()
    for cmd in found:
        if cmd not in seen:
            deduped.append(cmd)
            seen.add(cmd)

    return deduped


def _safe_text_len(value):
    if value is None:
        return 0
    if isinstance(value, str):
        return len(value)
    return len(str(value))


def output_size_check(outputs, max_len=500 * 1024):
    total_size = sum(
        _safe_text_len(getattr(out, "stdout", ""))
        + _safe_text_len(getattr(out, "stderr", ""))
        for out in outputs
    )
    return total_size, total_size >= max_len


rows = []
total_unpickled_size = 0

for name in sorted(os.listdir(cache_dir)):
    path = os.path.join(cache_dir, name)
    if not os.path.isfile(path):
        continue

    file_size = os.path.getsize(path)
    created_ts = datetime.fromtimestamp(os.path.getctime(path)).isoformat(
        sep=" ", timespec="seconds"
    )

    commands = []
    command_source = "none"
    command_preview = "<unknown>"
    unpickled_size = 0
    object_type = "<unknown>"
    output_total_size = 0
    output_too_large_hit = False

    try:
        with open(path, "rb") as fh:
            obj = pickle.load(fh)

        object_type = type(obj).__name__
        commands = extract_commands(obj)

        outputs = getattr(obj, "outputs", None)
        if isinstance(outputs, (list, tuple)):
            output_total_size, output_too_large_hit = output_size_check(outputs)

        if commands:
            command_source = "extracted"
            # Keep preview compact
            command_preview = " | ".join(commands[:3])
            if len(commands) > 3:
                command_preview += f" | ... (+{len(commands) - 3} more)"
        else:
            command_source = "missing-fields"

        unpickled_size = deep_getsizeof(obj)
        total_unpickled_size += unpickled_size

    except Exception as e:
        command_source = "unpickle-failed"
        command_preview = f"<failed to unpickle: {e}>"

    rows.append(
        {
            "file": name,
            "filesize": file_size,
            "filesize_h": fmt_bytes(file_size),
            "unpickled_size": unpickled_size,
            "unpickled_size_h": fmt_bytes(unpickled_size),
            "created": created_ts,
            "object_type": object_type,
            "n_commands": len(commands),
            "command_source": command_source,
            "command_preview": command_preview,
            "output_total_size": output_total_size,
            "output_total_size_h": fmt_bytes(output_total_size),
            "output_too_large_hit": output_too_large_hit,
            "commands": commands,
        }
    )


df = pd.DataFrame(rows).sort_values("filesize", ascending=False).reset_index(drop=True)

# Show compact table first (nice in notebook)
summary_cols = [
    "file",
    "filesize_h",
    "unpickled_size_h",
    "created",
    "object_type",
    "n_commands",
    "output_total_size_h",
    "output_too_large_hit",
    "command_source",
    "command_preview",
]
display(df[summary_cols])

print("\nTotal files:", len(df))
print("Total unpickled size:", fmt_bytes(total_unpickled_size))
print("Files with extracted commands:", int((df["n_commands"] > 0).sum()))
print(
    "Files with missing command fields:",
    int((df["command_source"] == "missing-fields").sum()),
)
print("Files that hit output size check:", int(df["output_too_large_hit"].sum()))

# Quick investigation output: show examples with no commands
missing = df[df["n_commands"] == 0]
if not missing.empty:
    print("\nExample files with no extracted commands:")
    display(missing[["file", "object_type", "command_source"]].head(10))

,file,filesize_h,unpickled_size_h,created,object_type,n_commands,output_total_size_h,output_too_large_hit,command_source,command_preview
0,3ff7ee36ce3e325791db5ad69ca3e92bad12594c9b2d2f...,108.02 KB,108.99 KB,2026-03-03 20:42:27,ShellCacheType,1,107.60 KB,False,extracted,cat tracing_output.log
1,c9c57cdce9f9660e91c1f9cb9f21f697c5d23606a95ec3...,103.60 KB,104.57 KB,2026-03-03 20:31:42,ShellCacheType,1,103.18 KB,False,extracted,cat tracing_output.log
2,fba07d2c963c54d5642efc1b7906d6e296825f2ec26c1f...,95.85 KB,97.81 KB,2026-03-03 22:32:07,ShellCacheType,4,95.17 KB,False,extracted,cat builder_impl.hpp | cat builder_impl.cpp | ...
3,ba0d45b27a1f2f8c8d702111b792592f02d0a12116190f...,76.89 KB,153.32 KB,2026-03-03 15:38:41,ShellCacheType,2,75.87 KB,False,extracted,"file db | sed -n '1,260p' db"
4,bc5f7929ae881c26fbb46e455fc32bd3e272b6cfcf51f0...,65.65 KB,67.62 KB,2026-03-03 23:04:40,ShellCacheType,4,64.99 KB,False,extracted,cat query_q3.cpp | cat query_q3.hpp | cat buil...
...,...,...,...,...,...,...,...,...,...,...
1029,8e2956e0e568173e1bc0aa174f256436098e32bbcfa887...,374.00 B,1.32 KB,2026-03-03 19:50:26,ShellCacheType,1,0.00 B,False,extracted,true
1030,9065567bc5a627ccc57eef306c8b1d96d82af9aa74a15d...,374.00 B,1.32 KB,2026-03-03 19:49:37,ShellCacheType,1,0.00 B,False,extracted,true
1031,81b6271512decfe791cc79717c390731dad01c6eb9d2dc...,374.00 B,1.32 KB,2026-03-03 19:49:09,ShellCacheType,1,0.00 B,False,extracted,true
1032,6a2e91e45b0f17addca4253832b206cec80239f2fdcde6...,374.00 B,1.32 KB,2026-03-03 19:50:15,ShellCacheType,1,0.00 B,False,extracted,true



Total files: 1034
Total unpickled size: 9.52 MB
Files with extracted commands: 1034
Files with missing command fields: 0
Files that hit output size check: 0


In [4]:
# filter df by timestamps after 2/20
filtered_df = df[df["created"] >= "2026-02-20"].reset_index(drop=True)
filtered_df

,file,filesize,filesize_h,unpickled_size,unpickled_size_h,created,object_type,n_commands,command_source,command_preview,output_total_size,output_total_size_h,output_too_large_hit,commands
0,3ff7ee36ce3e325791db5ad69ca3e92bad12594c9b2d2f...,110609,108.02 KB,111610,108.99 KB,2026-03-03 20:42:27,ShellCacheType,1,extracted,cat tracing_output.log,110186,107.60 KB,False,[cat tracing_output.log]
1,c9c57cdce9f9660e91c1f9cb9f21f697c5d23606a95ec3...,106082,103.60 KB,107083,104.57 KB,2026-03-03 20:31:42,ShellCacheType,1,extracted,cat tracing_output.log,105659,103.18 KB,False,[cat tracing_output.log]
2,fba07d2c963c54d5642efc1b7906d6e296825f2ec26c1f...,98146,95.85 KB,100162,97.81 KB,2026-03-03 22:32:07,ShellCacheType,4,extracted,cat builder_impl.hpp | cat builder_impl.cpp | ...,97459,95.17 KB,False,"[cat builder_impl.hpp, cat builder_impl.cpp, c..."
3,ba0d45b27a1f2f8c8d702111b792592f02d0a12116190f...,78734,76.89 KB,157003,153.32 KB,2026-03-03 15:38:41,ShellCacheType,2,extracted,"file db | sed -n '1,260p' db",77686,75.87 KB,False,"[file db, sed -n '1,260p' db]"
4,bc5f7929ae881c26fbb46e455fc32bd3e272b6cfcf51f0...,67230,65.65 KB,69246,67.62 KB,2026-03-03 23:04:40,ShellCacheType,4,extracted,cat query_q3.cpp | cat query_q3.hpp | cat buil...,66547,64.99 KB,False,"[cat query_q3.cpp, cat query_q3.hpp, cat build..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029,8e2956e0e568173e1bc0aa174f256436098e32bbcfa887...,374,374.00 B,1347,1.32 KB,2026-03-03 19:50:26,ShellCacheType,1,extracted,true,0,0.00 B,False,[true]
1030,9065567bc5a627ccc57eef306c8b1d96d82af9aa74a15d...,374,374.00 B,1347,1.32 KB,2026-03-03 19:49:37,ShellCacheType,1,extracted,true,0,0.00 B,False,[true]
1031,81b6271512decfe791cc79717c390731dad01c6eb9d2dc...,374,374.00 B,1347,1.32 KB,2026-03-03 19:49:09,ShellCacheType,1,extracted,true,0,0.00 B,False,[true]
1032,6a2e91e45b0f17addca4253832b206cec80239f2fdcde6...,374,374.00 B,1347,1.32 KB,2026-03-03 19:50:15,ShellCacheType,1,extracted,true,0,0.00 B,False,[true]
